# **HW9 Model Merging**
This is the Colab Notebook of ML2025 HW9.


# Check GPU Availability

In [ ]:
# Check GPU Availability
!nvidia-smi

# Install Packages

In [ ]:
# 安裝 PyTorch
# 2026 版本注意：Colab 預設 CUDA 可能已升至 12.6+
# 若 cu124 安裝失敗，改用 cu126 或直接用 Colab 預裝的 torch
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-c", "import torch; print(torch.__version__)"],
    capture_output=True, text=True
)
existing_version = result.stdout.strip()
print(f"現有 torch 版本：{existing_version}")

# 若版本不符則安裝指定版本（cu124）
if not existing_version.startswith("2.5"):
    !pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 \
        --index-url https://download.pytorch.org/whl/cu124 -q
else:
    print("torch 版本符合，跳過安裝")

In [ ]:
# Install Packages
!pip install datasets==3.5.1 -q

In [ ]:
# 安裝 bitsandbytes
# 2026 版本注意：0.45.5 若與新版 CUDA 不相容，改用 >=0.45.5
!pip install "bitsandbytes>=0.45.5" -q

# Load Dataset from huggingface
Save GSM8K.json and ARC.json on Colab for inference.

In [ ]:
# Load Dataset from huggingface
from datasets import load_dataset
import json

dataset = load_dataset("MonicaHuang/ML2025_HW9")

gsm8k = dataset["GSM8K"]
arc = dataset["ARC"]
gsm8k_path = "/content/GSM8K.json"
arc_path = "/content/ARC.json"
gsm8k_list = gsm8k.to_list()
arc_list = arc.to_list()

with open(gsm8k_path, "w") as f:
    json.dump(gsm8k_list, f, indent=2)

with open(arc_path, "w") as f:
    json.dump(arc_list, f, indent=2)

print(f"GSM8K: {len(gsm8k_list)} 筆，ARC: {len(arc_list)} 筆")

# 安裝修改後的 peft 套件（TA 版本 + SCE 修改）

**步驟：**
1. 從 Google Drive 下載 TA 版 peft zip
2. 解壓縮到 Google Drive 的指定資料夾
3. 將本次作業修改的 `merge_utils.py` 和 `model.py` 覆蓋到 peft 套件內
4. 以 editable mode 安裝

In [ ]:
# 掛載 Google Drive
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/ml2025_hw9

In [ ]:
# 下載並解壓縮 TA 版 peft 套件
!pip install -U gdown -q
!gdown --id 1HK8q4l7aMI6MjNdeJyzLCqbgM8lZCAZV -O /content/peft.zip
!unzip -q /content/peft.zip -d /content/drive/MyDrive/ml2025_hw9
print("解壓縮完成")

In [ ]:
# ============================================================
# 將 SCE 實作寫入 peft 套件的 merge_utils.py
# 這個 cell 會把 SCE 相關函數 append 到 merge_utils.py 末尾
#
# ⚠️ 必須先執行上方的「下載/解壓縮 cell」才能跑這個 cell！
# ============================================================
import os, glob

# 自動偵測解壓後的 peft 根目錄（zip 解出的資料夾名稱可能不固定）
BASE_DIR = "/content/drive/MyDrive/ml2025_hw9"
candidates = glob.glob(f"{BASE_DIR}/peft*")
if not candidates:
    raise FileNotFoundError(
        f"找不到 peft 目錄！請先執行上方的下載/解壓縮 cell。\n"
        f"搜尋路徑：{BASE_DIR}"
    )
PEFT_ROOT = candidates[0]
print(f"偵測到 peft 根目錄：{PEFT_ROOT}")

MERGE_UTILS_PATH = f"{PEFT_ROOT}/src/peft/utils/merge_utils.py"
MODEL_PATH = f"{PEFT_ROOT}/src/peft/tuners/lora/model.py"

# 確認關鍵檔案存在
for p in [MERGE_UTILS_PATH, MODEL_PATH]:
    if not os.path.exists(p):
        raise FileNotFoundError(
            f"找不到：{p}\n"
            f"請確認 peft 套件結構正確，或改用其他 Google Drive 連結重新下載。"
        )

# ── 新增 SCE 函數到 merge_utils.py ──
sce_code = '''

# ============================================================
# SCE (Select-Calculate-Erase) 合併演算法
# 論文：FuseChat: Knowledge fusion of chat models (arXiv:2408.07990)
# ============================================================

def sce_mask(
    task_tensors: torch.Tensor,
    density: float,
    mask_dtype: Optional[torch.dtype] = None,
) -> torch.Tensor:
    """S 步驟：跨所有 task vector 選 top-k variance 的位置。"""
    if density <= 0:
        return torch.zeros_like(task_tensors[0], dtype=mask_dtype)
    if density >= 1:
        return torch.ones_like(task_tensors[0], dtype=mask_dtype)

    # 計算每個參數位置在 T 個 task 上的 variance
    var = torch.var(task_tensors, dim=0, unbiased=False)  # shape: (...)
    nonzero = torch.count_nonzero(var)
    k = int(nonzero.item() * density)
    if k == 0:
        return torch.zeros_like(var, dtype=mask_dtype)

    _, indices = torch.topk(var.abs().view(-1), k=k, largest=True)
    mask = torch.zeros_like(var, dtype=mask_dtype)
    mask.view(-1)[indices] = 1
    return mask


def sce_weight(task_tensors: torch.Tensor) -> torch.Tensor:
    """C 步驟：計算每個 task 的平方能量係數 η。"""
    weights = torch.mean(
        task_tensors ** 2,
        dim=list(range(1, task_tensors.dim()))
    )  # shape: (T,)
    weight_sum = torch.sum(weights).item()
    if abs(weight_sum) < 1e-6:
        return torch.ones_like(weights) / weights.shape[0]
    return weights / weight_sum


def sce(
    task_tensors: List[torch.Tensor],
    density: float,
    majority_sign_method: Literal["total", "frequency"] = "total",
) -> torch.Tensor:
    """SCE 完整合併：S → E → C → 加權求和。"""
    tv = torch.stack(task_tensors, dim=0)  # (T, ...)

    # S：選出 variance 最高的位置
    if density < 1.0:
        mask = sce_mask(tv, density, mask_dtype=tv.dtype)
        tv = tv * mask.unsqueeze(0)

    # E：消除少數符號方向
    erase_mask = sign_consensus_mask(tv, method=majority_sign_method, mask_dtype=tv.dtype)
    tv = tv * erase_mask

    # C：計算各 task 的能量係數
    tv_weights = sce_weight(tv)  # shape: (T,)

    # 加權求和
    weight_shape = [-1] + [1] * (tv.dim() - 1)
    merged = torch.sum(tv * tv_weights.view(weight_shape), dim=0)
    return merged
'''

# 避免重複 append（重複執行 cell 時的保護）
with open(MERGE_UTILS_PATH, 'r', encoding='utf-8') as f:
    existing = f.read()

if 'def sce(' in existing:
    print("merge_utils.py 已含 SCE，跳過 append")
else:
    with open(MERGE_UTILS_PATH, 'a', encoding='utf-8') as f:
        f.write(sce_code)
    print("merge_utils.py 已更新（SCE 函數已加入）")

In [ ]:
# ============================================================
# 修改 model.py，加入 SCE 的 import 和 dispatch 邏輯
# PEFT_ROOT / MODEL_PATH 由上一個 cell 設定，必須按順序執行
# ============================================================

with open(MODEL_PATH, 'r', encoding='utf-8') as f:
    content = f.read()

# 避免重複修改（重複執行 cell 時的保護）
if 'combination_type == "sce"' in content:
    print("model.py 已含 SCE 修改，跳過")
else:
    modified = False

    # 修改 1：import 加入 sce
    old_import = "from peft.utils.merge_utils import magnitude_prune, ties"
    new_import = "from peft.utils.merge_utils import magnitude_prune, ties, sce"
    if old_import in content:
        content = content.replace(old_import, new_import)
        print("修改 1 完成：import sce")
        modified = True
    else:
        print("⚠️ 找不到 import 位置，請手動確認 model.py 的 import 格式")

    # 修改 2：combination_type 合法清單加入 "sce"
    old_list = '"linear", "ties", "dare_linear", "dare_ties", "magnitude_prune"'
    new_list = '"linear", "ties", "dare_linear", "dare_ties", "magnitude_prune", "sce"'
    if old_list in content:
        content = content.replace(old_list, new_list)
        print("修改 2 完成：combination_type 清單")
        modified = True
    else:
        print("⚠️ 找不到 combination_type 清單，請手動確認")

    # 修改 3：dispatch 邏輯加入 sce（使用寬鬆匹配，容許縮排差異）
    import re
    pattern = r'(elif combination_type == "ties":\s+lora_deltas\[i\] = ties\(task_tensors, valid_weights, density, majority_sign_method\))'
    replacement = (
        r'\1'
        '\n            elif combination_type == "sce":'
        '\n                lora_deltas[i] = sce(task_tensors, density, majority_sign_method)'
    )
    new_content, n = re.subn(pattern, replacement, content)
    if n > 0:
        content = new_content
        print("修改 3 完成：dispatch 邏輯")
        modified = True
    else:
        print("⚠️ 找不到 ties dispatch，請在 ties 的 elif 後面手動加入：")
        print('    elif combination_type == "sce":')
        print('        lora_deltas[i] = sce(task_tensors, density, majority_sign_method)')

    if modified:
        with open(MODEL_PATH, 'w', encoding='utf-8') as f:
            f.write(content)
        print("\nmodel.py 已更新")

In [ ]:
# 以 editable mode 安裝修改後的 peft 套件
# PEFT_ROOT 由前面的 patch cell 自動偵測設定

%cd {PEFT_ROOT}
!pip install -e . -q
%cd /content

import sys
peft_src = f"{PEFT_ROOT}/src"
if peft_src not in sys.path:
    sys.path.insert(0, peft_src)

print(f"peft 安裝完成，src 路徑：{peft_src}")

# Import Packages

In [ ]:
# 設定 cuBLAS 確定性行為（需在 import torch 前設定）
%env CUBLAS_WORKSPACE_CONFIG=:4096:8

In [ ]:
import json
import itertools
import torch
import os
import re
import numpy as np
import string

from tqdm import tqdm
from peft import PeftModel
from transformers import (
    GenerationConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    logging as hf_logging,
)
from peft import LoraConfig
from transformers import BitsAndBytesConfig

# 抑制 transformers 的非必要 warning（2026 版本較多廢棄警告）
hf_logging.set_verbosity_error()

print(f"torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Prompt Generation

> You're not allowed to modify instructions, questions and options in HW9!


In [ ]:
instruction_dict = {
    "GSM8K": 'You are given a math question and four answer options (associated with "A", "B", "C", "D"). Your task is to carefully analyze the problem, apply logical reasoning, and select the correct answer. There is only one correct answer for each question.',
    "ARC": 'You are given a science question and four answer options (associated with "A", "B", "C", "D"). Your task is to find the correct answer based on scientific facts, knowledge, and reasoning. There is only one correct answer for each question.',
}

def mcqa_prompt(task_name, instruction, question, options):
    sys_msg = instruction_dict[task_name]
    options_dict = options
    IDs = ['A', 'B', 'C', 'D', 'E', 'F']
    if "D" in options_dict:
        op_ids = [IDs[i] for i in range(4)]
    else:
        op_ids = [IDs[i] for i in range(3)]
    options_list = [options_dict[op_id] for op_id in op_ids]

    user_prompt = f"Question: {question}; Options: " + \
        " ".join([f"({op_id}) {option}" for op_id, option in zip(op_ids, options_list)])

    return f"""[INST] <<SYS>>You are a helpful assistant and good at solving tasks based on task instructions and inputs.<</SYS>> Instruction: {sys_msg} Input: {user_prompt}[/INST]"""

def generate_prompt(task_name, tokenizer, data_point):
    return mcqa_prompt(
        task_name=task_name,
        instruction=data_point["instruction"],
        question=data_point["question"],
        options=data_point["options"],
    )

# Seed Settings

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = '0'
device = torch.device('cuda:0')

In [ ]:
torch.manual_seed(42)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.benchmark = False
np.random.seed(42)

# Model Configs

In [ ]:
# 不要修改！
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)
lora_config = LoraConfig(
    r=8,
    target_modules=["q_proj", 'k_proj', "v_proj"],
    task_type="CAUSAL_LM",
    lora_alpha=16,
    lora_dropout=0.05
)
RANDOM_SEED = 42
CUTOFF_LEN = 700

# Load Base Model and Adapters
LoRA weights 從 HuggingFace 載入。

In [ ]:
print("載入 Model 和 Tokenizer ...")

base_model_name = "unsloth/llama-2-7b-chat-bnb-4bit"
tokenizer = AutoTokenizer.from_pretrained(
    base_model_name, quantization_config=quant_config, use_fast=True
)
tokenizer.pad_token_id = 1
tokenizer.eos_token_id = 2

model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=quant_config,
    low_cpu_mem_usage=True,
    torch_dtype=torch.float16
).to(device)
model.resize_token_embeddings(len(tokenizer))

MERGE_TASK_NAMES = ["GSM8K", "ARC"]
hf_model_dict = {
    "GSM8K": "MonicaHuang/llama-2-7b-chat-GSM8K-MCQA",
    "ARC": "chenjoachim/llama-2-7b-chat-ARC-MCQA",
}

# 載入兩個 LoRA adapter（task vector）
adapters = MERGE_TASK_NAMES
model = PeftModel.from_pretrained(
    model, hf_model_dict[adapters[0]], adapter_name=adapters[0]
).to(device)
for adapter in adapters[1:]:
    _ = model.load_adapter(hf_model_dict[adapter], adapter_name=adapter, device=device)

print("Model 載入完成")

# 推論輔助函數

In [ ]:
OUTPUT_DIR = "outputs"

def run_inference(task_name, model, tokenizer, generation_config, hyperparameters):
    """對指定任務跑完整推論，回傳 results list。"""
    test_set_path = f"/content/{task_name}.json"
    print(f"載入測試資料 {task_name} ...")
    with open(test_set_path, "r", encoding="utf-8") as f:
        test_datas = json.load(f)

    results = []
    for test_data in tqdm(test_datas, desc=f"{task_name} 推論"):
        prompt = generate_prompt(task_name, tokenizer, test_data)
        inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=True)
        input_ids = inputs["input_ids"].cuda()

        model.eval()
        with torch.no_grad():
            generation_output = model.generate(
                input_ids=input_ids,
                generation_config=generation_config,
                return_dict_in_generate=True,
                output_scores=True,
                max_new_tokens=hyperparameters["max_new_len"],
            )

        for s in generation_output.sequences:
            predict = tokenizer.decode(s)

        response = predict.split("[/INST]")[-1].split('</s>')[0].strip()
        results.append({"id": test_data["id"], "response": response})

    return results


def save_results(results, merge_type, task_name, weights, density):
    """將推論結果存到對應路徑。"""
    if weights:
        wnames = '_'.join([str(float(w)) for w in weights])
    else:
        wnames = 'auto'
    result_dir = f"{OUTPUT_DIR}/{merge_type}/{task_name}"
    os.makedirs(result_dir, exist_ok=True)
    out_path = f"{result_dir}/w_{wnames}_d_{density}_result.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    return out_path


def apply_merge(model, adapters, merge_type, weights, density):
    """套用指定的合併演算法，回傳 merge adapter name。"""
    adapter_name = "merge"

    # 若上一輪已有 merge adapter，先刪掉
    if adapter_name in model.peft_config:
        model.delete_adapter(adapter_name)

    if merge_type == "linear":
        model.add_weighted_adapter(
            adapters, weights, adapter_name, combination_type="linear"
        )
    elif merge_type == "magnitude_prune":
        model.add_weighted_adapter(
            adapters, weights, adapter_name,
            combination_type="magnitude_prune", density=density
        )
    elif merge_type == "ties":
        model.add_weighted_adapter(
            adapters, weights, adapter_name,
            combination_type="ties", density=density
        )
    elif merge_type == "dare_linear":
        model.add_weighted_adapter(
            adapters, weights, adapter_name,
            combination_type="dare_linear", density=density
        )
    elif merge_type == "dare_ties":
        model.add_weighted_adapter(
            adapters, weights, adapter_name,
            combination_type="dare_ties", density=density
        )
    elif merge_type == "sce":
        # SCE 不使用外部 weights（自動計算 η），density 控制 S 步驟
        model.add_weighted_adapter(
            adapters, [1.0] * len(adapters), adapter_name,
            combination_type="sce", density=density
        )
    else:
        raise ValueError(f"未知的 merge_type: {merge_type}")

    model.set_adapter(adapter_name)
    return adapter_name

print("推論輔助函數定義完成")

---

# 單次推論（手動設定參數）

> **規則**：兩個任務必須使用完全相同的 merging setting！
> 推論完整 400 題預估需 2~4 小時（T4 GPU）。

In [ ]:
# ================================================================
# TODO：調整這裡的參數後執行以下的推論 cell
# ================================================================
MERGE_TYPE = "sce"       # 選項: "linear", "magnitude_prune", "ties",
                         #       "dare_linear", "dare_ties", "sce"
density = 0.5            # 用於 magnitude_prune / ties / dare_* / sce 的剪枝比例
weight_math = 1.0        # GSM8K 的權重（SCE 不使用此欄位）
weight_science = 1.0     # ARC 的權重（SCE 不使用此欄位）

weights = [weight_math, weight_science] if MERGE_TYPE != "sce" else None

In [ ]:
# Generation Config
# TODO：可以調整這裡的解碼策略
hyperparameters = {
    "do_sample": False,
    "temperature": None,
    "num_beams": 1,
    "top_p": None,
    "no_repeat_ngram_size": 3,
    "max_new_len": 400,
}
generation_config = GenerationConfig(
    do_sample=hyperparameters["do_sample"],
    temperature=hyperparameters["temperature"],
    top_p=hyperparameters["top_p"],
    num_beams=hyperparameters["num_beams"],
    pad_token_id=1,
    max_new_tokens=hyperparameters["max_new_len"],
)
print("Generation Config 設定完成")

In [ ]:
# 套用合併演算法
apply_merge(model, adapters, MERGE_TYPE, weights, density)
print(f"合併完成：{MERGE_TYPE}, density={density}, weights={weights}")

# 推論兩個任務
for task in ["GSM8K", "ARC"]:
    results = run_inference(task, model, tokenizer, generation_config, hyperparameters)
    path = save_results(results, MERGE_TYPE, task, weights, density)
    print(f"結果儲存至：{path}")

---

# 超參數網格搜尋

系統性跑多種 (algorithm, density, weights) 組合，找出最佳設定。

⚠️ 注意：每組約 2~4 小時，建議在時間充裕時分批執行。

In [ ]:
# ================================================================
# 超參數網格搜尋設定
# 可依時間調整搜尋範圍
# ================================================================

GRID = [
    # (merge_type, density, [weight_math, weight_science])
    # --- Task Arithmetic（linear）---
    ("linear",          1.0, [1.0, 1.0]),
    ("linear",          1.0, [1.0, 0.5]),
    # --- Magnitude Prune ---
    ("magnitude_prune", 0.2, [1.0, 1.0]),
    ("magnitude_prune", 0.5, [1.0, 1.0]),
    ("magnitude_prune", 0.8, [1.0, 1.0]),
    # --- TIES ---
    ("ties",            0.3, [1.0, 1.0]),
    ("ties",            0.5, [1.0, 1.0]),
    ("ties",            0.7, [1.0, 1.0]),
    # --- DARE Linear ---
    ("dare_linear",     0.5, [1.0, 1.0]),
    # --- DARE TIES ---
    ("dare_ties",       0.5, [1.0, 1.0]),
    # --- SCE（自動計算 η，density 控制 S 步驟）---
    ("sce",             0.3, None),
    ("sce",             0.5, None),
    ("sce",             0.7, None),
    ("sce",             1.0, None),  # density=1.0 等於跳過 S 步驟
]

# Generation Config（網格搜尋期間固定使用 greedy decoding）
grid_hyperparameters = {
    "do_sample": False,
    "temperature": None,
    "num_beams": 1,
    "top_p": None,
    "no_repeat_ngram_size": 3,
    "max_new_len": 300,  # 網格搜尋時縮短以節省時間
}
grid_generation_config = GenerationConfig(
    do_sample=grid_hyperparameters["do_sample"],
    temperature=grid_hyperparameters["temperature"],
    top_p=grid_hyperparameters["top_p"],
    num_beams=grid_hyperparameters["num_beams"],
    pad_token_id=1,
    max_new_tokens=grid_hyperparameters["max_new_len"],
)

print(f"共 {len(GRID)} 組設定待搜尋")

In [ ]:
# 執行網格搜尋（可中斷後從特定 index 繼續）
START_FROM = 0  # 如果中途中斷，從這個 index 繼續

for idx, (merge_type, density_val, weights_val) in enumerate(GRID):
    if idx < START_FROM:
        print(f"[{idx+1}/{len(GRID)}] 跳過 {merge_type} d={density_val}")
        continue

    print(f"\n{'='*60}")
    print(f"[{idx+1}/{len(GRID)}] {merge_type}, density={density_val}, weights={weights_val}")
    print('='*60)

    try:
        apply_merge(model, adapters, merge_type, weights_val, density_val)

        for task in ["GSM8K", "ARC"]:
            results = run_inference(
                task, model, tokenizer, grid_generation_config, grid_hyperparameters
            )
            path = save_results(results, merge_type, task, weights_val, density_val)
            print(f"  → 儲存：{path}")

    except Exception as e:
        print(f"  ⚠️ 錯誤：{e}，跳過此組")
        continue

print("\n網格搜尋完成！")

---

# 產生 Judgeboi 提交檔案（pred.json）

選擇表現最好的設定，合併 GSM8K 和 ARC 的結果產生 pred.json。

**注意：兩個任務必須來自同一組 merging setting！**

In [ ]:
# TODO：填入要提交的最佳設定對應的路徑
BEST_MERGE_TYPE = "sce"
BEST_DENSITY    = 0.5
BEST_WEIGHTS    = None  # SCE 使用 None；其他演算法填 [w_math, w_science]

if BEST_WEIGHTS:
    wnames = '_'.join([str(float(w)) for w in BEST_WEIGHTS])
else:
    wnames = 'auto'

arc_path   = f"{OUTPUT_DIR}/{BEST_MERGE_TYPE}/ARC/w_{wnames}_d_{BEST_DENSITY}_result.json"
gsm8k_path = f"{OUTPUT_DIR}/{BEST_MERGE_TYPE}/GSM8K/w_{wnames}_d_{BEST_DENSITY}_result.json"
output_path = "pred.json"

def load_list_as_dict(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return {item['id']: item['response'] for item in data if 'id' in item and 'response' in item}

gsm8k_dict = load_list_as_dict(gsm8k_path)
arc_dict   = load_list_as_dict(arc_path)
combined   = {**gsm8k_dict, **arc_dict}

print(f"GSM8K: {len(gsm8k_dict)} 筆，ARC: {len(arc_dict)} 筆，共 {len(combined)} 筆")

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(combined, f, indent=2, ensure_ascii=False)

print(f"pred.json 已儲存：{output_path}")

## End of HW9 Notebook
Thank you for reading!

In [ ]:
# Appendix: Batch Decoding/Generation
'''
def evaluate_batch(task_name, model, tokenizer, data_points, generation_config, max_len, verbose=True):

    prompts = [generate_prompt(task_name, tokenizer, dp) for dp in data_points]

    # Batch tokenization
    inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=CUTOFF_LEN)
    input_ids = inputs["input_ids"].cuda()

    model.eval()
    with torch.no_grad():
        generation_output = model.generate(
            input_ids=input_ids,
            generation_config=generation_config,
            return_dict_in_generate=True,
            output_scores=True,
            max_new_tokens=max_len,
        )

    outputs = tokenizer.batch_decode(generation_output.sequences, skip_special_tokens=True)

    if verbose:
        for idx, output in enumerate(outputs):
            print("================= Response of Model ==================")
            print({
                "input_prompt": prompts[idx],
                "output_id": generation_output.sequences[idx],
                "output": output,
            })

    return outputs

tokenizer.padding_side = "left"
config = {
    "batch_size": 4
}
results = []
max_iteration = len(test_datas)
BATCH_SIZE = config["batch_size"]

for batch_start in tqdm(range(0, max_iteration, BATCH_SIZE), total=(max_iteration // BATCH_SIZE)+1):
    batch_data = test_datas[batch_start:batch_start+BATCH_SIZE]

    predicts = evaluate_batch(
        TEST_TASK,
        model,
        tokenizer,
        batch_data,
        generation_config,
        hyperparameters["max_new_len"],
        verbose = False
    )

    for predict, test_data in zip(predicts, batch_data):
        response = predict.split("[/INST]")[-1].split('</s>')[0].strip()
        results.append({
            "id": test_data["id"],
            "response": response
        })
'''